In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GatedGraphConv, global_mean_pool

class TemporalGraphNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super(TemporalGraphNetwork, self).__init__()
        self.num_layers = num_layers
        self.convs = nn.ModuleList([GatedGraphConv(input_dim, hidden_dim)])
        for _ in range(num_layers - 1):
            self.convs.append(GatedGraphConv(hidden_dim, hidden_dim))
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index, edge_attr)
            x = F.relu(x)
        x = global_mean_pool(x, batch)
        x = self.fc(x)
        return F.log_softmax(x, dim=1)




In [ ]:
import networkx


# Assuming you have a list of sequences, each containing a list of graphs
sequences = []  # Your list of sequences, each containing a list of graphs
group_dict = {'ATL': (26, 3), 'Climp': (31, 2), 'Control': (31, 0), 'RTN': (29, 1)}
for seq in range(1, 32):
    try:
        for i in range(100):
            skel = imageio.imread(f'/localhome/asa420/MIAL/data/confocal-data/{group}/skel/A1_decon_t000_ch00_skel.png')



dataset = []

for sequence in sequences:
    for graph_data in sequence:
        edge_index = torch.tensor(graph_data.adjacency_matrix.nonzero(), dtype=torch.long)
        x = torch.ones(graph_data.adjacency_matrix.shape[0], 1)  # Node features (e.g., all ones)
        edge_attr = graph_data.edge_features  # Replace with your edge features if any
        data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
        dataset.append(data)



In [ ]:
loader = DataLoader(dataset, batch_size=1, shuffle=True)

# Create an instance of the TemporalGraphNetwork
input_dim = 1  # Change this based on your node features
hidden_dim = 64
output_dim = 4  # Number of classes
num_layers = 2  # Number of GatedGraphConv layers

model = TemporalGraphNetwork(input_dim, hidden_dim, output_dim, num_layers)

# Define your loss function and optimizer
criterion = nn.NLLLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
num_epochs = 100

for epoch in range(num_epochs):
    total_loss = 0
    for data in loader:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, data.y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}, Loss: {total_loss}")

# After training, you can use the model for classification tasks on new temporal sequences.